# RHI Live Runtime v25 — Core Agent Kernel

Δ **Purpose:** consolidate the RHI work into one coherent AI runtime.

$$
\boxed{\text{Goal}=\text{new AI runtime, not H-probing}}
$$

v25 returns to the main branch:

$$
Q \rightarrow P_Q \rightarrow C_Q \rightarrow T \rightarrow B_i \rightarrow A_i \rightarrow G_{\Psi/\Omega} \rightarrow S \rightarrow A_{\text{final}}
$$

Where:

- $Q$ = user prompt
- $P_Q$ = task profile
- $C_Q$ = runtime contract / need-slot
- $T$ = trace continuity layer
- $B_i$ = branch candidates
- $A_i$ = task-local audits
- $G_{\Psi/\Omega}$ = collapse gate
- $S$ = payload shaper
- $A_{\text{final}}$ = final accepted answer

This notebook writes exactly two files:

1. `rhi_v25_<run_id>_bundle.json`
2. `rhi_v25_<run_id>_summary.csv`

No per-prompt fan-out. No H branch. No fallback $\Psi$ without real model output.


In [ ]:

from __future__ import annotations

import os, re, sys, json, math, uuid, time, random, traceback, subprocess, importlib
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUT_DIR = ROOT / "rhi_v25_outputs"
OUT_DIR.mkdir(exist_ok=True)

RUN_ID = "rhi_v25_" + uuid.uuid4().hex[:10]
SEED = 25
random.seed(SEED)
np.random.seed(SEED)

MODEL_ID_OR_PATH = os.environ.get("RHI_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
LOAD_REAL_MODEL = True
REQUIRE_MODEL_FOR_PSI = True
AUTO_INSTALL_MISSING_DEPS = True

RUN_PROMPT_LIMIT = 42
MAX_RECURSION_DEPTH = 2
BRANCH_ROLES = ["construct", "verify", "repair", "counter"]
MAX_NEW_TOKENS_BRANCH = 190
MAX_NEW_TOKENS_SHAPER = 140
TEMPERATURE = 0.45
SHAPER_ENABLED = True

print("RHI v25 Core Agent Kernel")
print("RUN_ID:", RUN_ID)
print("OUT_DIR:", OUT_DIR)
print("MODEL:", MODEL_ID_OR_PATH)


## 1. Runtime Data Structures

In [ ]:

@dataclass
class TraceEvent:
    t: int
    event_type: str
    payload: Dict[str, Any]

@dataclass
class RuntimeContract:
    profile: str
    prompt: str
    inverse_need: str
    preserved_function: str
    required_operations: List[str]
    required_terms: List[str]
    forbidden_terms: List[str]
    semantic_locks: Dict[str, str]
    success_criteria: List[str]
    failure_criteria: List[str]
    allowed_side_effects: List[str] = field(default_factory=list)
    rollback_plan: str = ""
    trace_update: str = ""

@dataclass
class BranchResult:
    role: str
    origin: str
    depth: int
    prompt_used: str
    answer: str
    score: float
    audit: Dict[str, Any]
    rejected: bool
    rejection_reasons: List[str]

@dataclass
class PromptResult:
    prompt: str
    profile: str
    contract: Dict[str, Any]
    state: str
    reason: str
    depth: int
    winner_branch: Optional[str]
    winner_origin: Optional[str]
    winner_score: float
    answer: str
    raw_winner_answer: str
    shaped_answer: Optional[str]
    shaping_attempted: bool
    shaping_accepted: bool
    branches: List[Dict[str, Any]]
    trace: List[Dict[str, Any]]
    metrics: Dict[str, Any]


## 2. Dependency Check and Model Load

A valid $\Psi$ requires real model output. Dependency or model failure becomes $\Omega_{\text{runtime}}$, not fake success.


In [ ]:

def _module_available(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False
    except Exception:
        return False

def ensure_runtime_dependencies() -> Dict[str, Any]:
    status = {"checked": True, "attempted_install": False, "missing_before": [], "missing_after": [], "errors": []}
    required = [("torch", "torch"), ("transformers", "transformers"), ("sentencepiece", "sentencepiece"), ("google.protobuf", "protobuf")]
    for module_name, pip_name in required:
        if not _module_available(module_name):
            status["missing_before"].append(pip_name)

    if status["missing_before"] and AUTO_INSTALL_MISSING_DEPS:
        status["attempted_install"] = True
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", *sorted(set(status["missing_before"]))], check=True)
            importlib.invalidate_caches()
        except Exception as e:
            status["errors"].append(repr(e))

    for module_name, pip_name in required:
        if not _module_available(module_name):
            status["missing_after"].append(pip_name)
    return status

DEPENDENCY_STATUS = ensure_runtime_dependencies()

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def get_device_info() -> Dict[str, Any]:
    info = {
        "torch_version": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "device_count": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
        "cuda_version": getattr(torch.version, "cuda", None),
    }
    if torch.cuda.is_available():
        info["gpu_name"] = torch.cuda.get_device_name(0)
    return info

DEVICE_INFO = get_device_info()

model = None
tokenizer = None
MODEL_READY = False
MODEL_GENERATION_READY = False
MODEL_ERROR = None
SMOKE_TEXT = None

def load_model():
    global model, tokenizer, MODEL_READY, MODEL_ERROR
    if not LOAD_REAL_MODEL:
        MODEL_READY = False
        MODEL_ERROR = "LOAD_REAL_MODEL=False"
        return
    try:
        print("Loading model:", MODEL_ID_OR_PATH)
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_OR_PATH)
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID_OR_PATH,
            dtype=dtype,
            device_map="auto" if torch.cuda.is_available() else None,
        )
        if not torch.cuda.is_available():
            model = model.to("cpu")
        model.eval()
        MODEL_READY = True
        print("MODEL_READY:", MODEL_READY, "DEVICE:", next(model.parameters()).device)
    except Exception as e:
        MODEL_ERROR = traceback.format_exc()
        MODEL_READY = False
        print("MODEL LOAD ERROR:", repr(e))

def model_smoke_test():
    global MODEL_GENERATION_READY, SMOKE_TEXT, MODEL_ERROR
    if not MODEL_READY:
        MODEL_GENERATION_READY = False
        return
    try:
        messages = [{"role": "user", "content": "Reply with READY only."}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(next(model.parameters()).device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=8, do_sample=False, return_dict_in_generate=True, pad_token_id=tokenizer.eos_token_id)
        gen = out.sequences[0][inputs.input_ids.shape[1]:]
        SMOKE_TEXT = tokenizer.decode(gen, skip_special_tokens=True).strip()
        MODEL_GENERATION_READY = bool(SMOKE_TEXT)
        print("MODEL_GENERATION_READY:", MODEL_GENERATION_READY)
        print("SMOKE:", SMOKE_TEXT)
    except Exception:
        MODEL_ERROR = traceback.format_exc()
        MODEL_GENERATION_READY = False
        print("SMOKE TEST FAILED")
        print(MODEL_ERROR)

load_model()
model_smoke_test()

print("DEPENDENCY_STATUS:", DEPENDENCY_STATUS)
print("DEVICE_INFO:", DEVICE_INFO)


## 3. Task Profiles and Semantic Carrier Locks

In [ ]:

PROFILE_THRESHOLDS = {
    "runtime_contract": 0.70,
    "tool_safety": 0.70,
    "evidence_control": 0.70,
    "state_recovery": 0.69,
    "memory_trace": 0.68,
    "inverse_retrieval": 0.68,
    "core_kernel": 0.70,
    "general": 0.72,
}

PROFILE_LEXICONS = {
    "runtime_contract": ["contract", "precondition", "postcondition", "success", "failure", "criteria", "side effect", "bounded", "function call", "api call", "before execution", "premature", "rollback plan", "runtime"],
    "tool_safety": ["tool call", "safe", "safety", "permission", "risk", "dangerous", "reject", "unsafe", "side effect", "override", "external api", "shell", "execute", "bounded risk", "allowed side effects"],
    "evidence_control": ["tool output", "tool result", "evidence", "observation", "command", "driver", "authority", "verify output", "hallucination", "retrieved evidence", "controller", "policy", "conflicting tool outputs", "contract gate"],
    "state_recovery": ["rollback", "recover", "recovery", "restore", "prior valid state", "failed", "mistake", "undo", "bad side effect", "trace", "causal trace", "corrupted", "state", "cascading failure"],
    "memory_trace": ["memory", "summary", "trace continuity", "causal", "state transitions", "observations", "decisions", "updates", "context amnesia", "across turns", "which-path", "live trace"],
    "inverse_retrieval": ["retrieval", "retrieve", "search", "keyword", "noun", "label", "title", "inverse", "missing slot", "shape-first", "function", "operation", "candidate", "rank", "surface term", "affordance", "preserved function", "downstream use"],
    "core_kernel": ["runtime interface", "model weights", "recursive collapse", "collapse control", "rhi kernel", "control field", "false psi", "honest omega", "payload shaping", "trace-governed"],
}

SEMANTIC_LOCKS = {
    "contract": "runtime execution contract, not legal agreement",
    "tool": "external action/function/API channel with possible side effects",
    "controller": "agent governance loop that owns policy and next-action selection",
    "policy": "runtime decision rule, not organizational ownership",
    "evidence": "observation submitted to verifier/controller, not a command",
    "memory": "causal trace continuity, not a paragraph summary",
    "retrieval": "operational fit / missing-slot recovery, not noun overlap",
    "shape": "operational structure, not literal circles/squares unless explicitly requested",
    "rollback": "restore prior valid state while preserving evidence/trace",
    "psi": "accepted collapse only when operational fit is preserved",
    "omega": "residue / no safe collapse; preferable to false acceptance",
}

def normalize_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip().lower())

def contains_any(text: str, terms: List[str]) -> bool:
    t = normalize_text(text)
    return any(term.lower() in t for term in terms)

def count_terms(text: str, terms: List[str]) -> int:
    t = normalize_text(text)
    return sum(1 for term in terms if term.lower() in t)

def classify_task_profile(prompt: str) -> Tuple[str, Dict[str, Any]]:
    p = normalize_text(prompt)
    scores = {profile: count_terms(p, terms) for profile, terms in PROFILE_LEXICONS.items()}

    if "tool output" in p or "tool result" in p or ("evidence" in p and ("tool" in p or "agent" in p)):
        scores["evidence_control"] += 4
    if "controller" in p and ("policy" in p or "tool" in p):
        scores["evidence_control"] += 3
    if "api call" in p and ("success" in p or "failure" in p or "criteria" in p):
        scores["runtime_contract"] += 4
    if "postcondition" in p or "precondition" in p:
        scores["runtime_contract"] += 3
    if "rollback" in p or "restore" in p or "recover" in p or "undo" in p:
        scores["state_recovery"] += 4
    if "memory" in p or "summary" in p or "trace continuity" in p:
        scores["memory_trace"] += 4
    if "retriev" in p or "search" in p or "keyword" in p or "noun" in p or "label" in p:
        scores["inverse_retrieval"] += 4
    if "safe" in p or "unsafe" in p or "permission" in p or "risk" in p or "side effect" in p:
        scores["tool_safety"] += 3
    if "rhi" in p or "collapse" in p or "recursive control" in p or "model weights" in p:
        scores["core_kernel"] += 4

    best_profile = max(scores, key=scores.get)
    if scores[best_profile] == 0:
        best_profile = "general"

    return best_profile, {"profile_scores": scores}


## 4. Runtime Contract Compiler

In [ ]:

def compile_contract(prompt: str, profile: str) -> RuntimeContract:
    base = {
        "profile": profile,
        "prompt": prompt,
        "inverse_need": "answer the prompt while preserving its operational intent",
        "preserved_function": "produce a useful answer that keeps the prompt's actual operation intact",
        "required_operations": ["identify task", "preserve intended operation", "avoid semantic drift"],
        "required_terms": [],
        "forbidden_terms": [],
        "semantic_locks": dict(SEMANTIC_LOCKS),
        "success_criteria": ["answer addresses the requested operation", "no wrong semantic carrier", "no unsupported collapse"],
        "failure_criteria": ["generic answer", "wrong domain", "surface vocabulary echo without operational fit"],
        "allowed_side_effects": [],
        "rollback_plan": "return Ω if operational fit cannot be established",
        "trace_update": "record profile, contract, branch audits, collapse reason, and payload decision",
    }

    if profile == "runtime_contract":
        base.update({
            "inverse_need": "define or explain a runtime execution contract before action",
            "preserved_function": "bound a tool/function/API call using preconditions, postconditions, success/failure criteria, side effects, rollback, and trace update",
            "required_operations": ["preconditions", "postconditions", "success/failure criteria", "bounded side effects", "rollback or safe failure", "trace update"],
            "required_terms": ["precondition", "postcondition", "success", "failure", "side effect", "rollback"],
            "forbidden_terms": ["legal agreement", "lawsuit", "liability", "stakeholder", "terms of service"],
        })
    elif profile == "tool_safety":
        base.update({
            "inverse_need": "decide whether a tool call is safe before acting",
            "preserved_function": "gate external action through permissions, preconditions, bounded side effects, risk scoring, and safe failure",
            "required_operations": ["permission check", "precondition check", "risk score", "bounded side effects", "safe failure", "rollback"],
            "required_terms": ["permission", "precondition", "risk", "side effect", "rollback"],
            "forbidden_terms": ["just run it", "execute first", "assume safe"],
        })
    elif profile == "evidence_control":
        base.update({
            "inverse_need": "treat tool output as evidence, not as the driver of the agent",
            "preserved_function": "route observations through controller policy, verifier, contract gate, and trace update before next action",
            "required_operations": ["treat output as observation", "verify evidence", "controller owns policy", "contract gate", "next-action selection"],
            "required_terms": ["evidence", "observation", "verify", "controller", "policy"],
            "forbidden_terms": ["tool decides", "output commands", "administrator owns policy", "security team owns policy"],
        })
    elif profile == "state_recovery":
        base.update({
            "inverse_need": "recover after failed action without losing causal trace",
            "preserved_function": "restore prior valid state while preserving evidence, trace, and rollback cause",
            "required_operations": ["detect failure", "freeze evidence", "restore prior valid state", "preserve causal trace", "prevent cascade"],
            "required_terms": ["rollback", "restore", "trace", "state", "failure"],
            "forbidden_terms": ["erase evidence", "forget trace", "start over blindly"],
        })
    elif profile == "memory_trace":
        base.update({
            "inverse_need": "explain or design memory as trace continuity",
            "preserved_function": "preserve causal order of states, observations, decisions, actions, results, rollbacks, and updates",
            "required_operations": ["state continuity", "observation history", "decision history", "update continuity", "which-path preservation"],
            "required_terms": ["trace", "state", "observation", "decision", "update"],
            "forbidden_terms": ["just a summary", "paragraph memory", "simple recap"],
        })
    elif profile == "inverse_retrieval":
        base.update({
            "inverse_need": "retrieve by inverse operational fit when noun/label matching fails",
            "preserved_function": "find the artifact that closes the missing operational slot, even if surface terms do not match",
            "required_operations": ["extract desired operation", "infer missing slot", "generate candidates by function", "reject noun-only matches", "verify preserved function"],
            "required_terms": ["operation", "function", "candidate", "verify", "missing"],
            "forbidden_terms": ["keyword only", "noun overlap only", "literal shape", "title match only"],
        })
    elif profile == "core_kernel":
        base.update({
            "inverse_need": "explain or design the RHI runtime as an inference-time AI control kernel",
            "preserved_function": "show how recursive contract field, branch audit, trace continuity, and Ψ/Ω collapse alter behavior without changing weights",
            "required_operations": ["task profile", "contract", "branch", "audit", "collapse", "trace", "payload"],
            "required_terms": ["runtime", "contract", "branch", "audit", "collapse", "trace"],
            "forbidden_terms": ["training weights changed", "fine-tuning required", "just a prompt"],
        })

    return RuntimeContract(**base)


## 5. Model Generation and Branch Engine

In [ ]:

def model_generate(prompt: str, max_new_tokens: int, temperature: float = TEMPERATURE) -> str:
    if REQUIRE_MODEL_FOR_PSI and not MODEL_GENERATION_READY:
        raise RuntimeError("Model is not generation-ready; refusing fallback model output.")
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(next(model.parameters()).device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            return_dict_in_generate=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out.sequences[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def contract_digest(contract: RuntimeContract) -> str:
    locks = "\n".join(f"- {k}: {v}" for k, v in contract.semantic_locks.items())
    return f"""TASK PROFILE: {contract.profile}
INVERSE NEED: {contract.inverse_need}
PRESERVED FUNCTION: {contract.preserved_function}

REQUIRED OPERATIONS:
{chr(10).join("- " + x for x in contract.required_operations)}

SEMANTIC LOCKS:
{locks}

FORBIDDEN DRIFT:
{chr(10).join("- " + x for x in contract.forbidden_terms)}

SUCCESS CRITERIA:
{chr(10).join("- " + x for x in contract.success_criteria)}
""".strip()

def make_branch_prompt(user_prompt: str, contract: RuntimeContract, role: str, prior_failure: Optional[Dict[str, Any]] = None) -> str:
    cd = contract_digest(contract)
    role_instructions = {
        "construct": "Construct the best direct answer. Preserve the operation. Do not recite the contract unless the user asked for a schema.",
        "verify": "Verify the operational need first. Reject wrong semantic carriers. Then answer.",
        "repair": "Repair the likely failure modes before answering. Keep the answer concise and operational.",
        "counter": "Stress-test the prompt for drift and answer only after preserving the intended operation.",
    }
    repair_block = ""
    if prior_failure:
        repair_block = f"""
PRIOR FAILURE SIGNAL:
- reason: {prior_failure.get("reason")}
- missing: {prior_failure.get("missing")}
- forbidden_hits: {prior_failure.get("forbidden_hits")}
- low_score_answer_preview: {prior_failure.get("answer_preview")}
""".strip()

    return f"""
You are running inside the RHI Core Agent Kernel.

USER PROMPT:
{user_prompt}

RUNTIME CONTRACT:
{cd}

BRANCH ROLE:
{role} — {role_instructions.get(role, role_instructions["construct"])}

{repair_block}

OUTPUT RULES:
- Answer the user directly.
- Preserve the preserved function.
- Use the semantic locks.
- Do not drift into forbidden meanings.
- Prefer Ω-style uncertainty over false confidence if the operation cannot be preserved.
""".strip()


## 6. Task-Local Audit and Collapse Gate

In [ ]:

LEGAL_DRIFT_TERMS = ["legal agreement", "lawsuit", "liability", "contract law", "terms of service", "stakeholder agreement"]
ADMIN_POLICY_DRIFT = ["system administrator", "security team", "organization's policy", "company policy", "administrator policy"]
TOOL_COMMAND_DRIFT = ["tool decides", "tool should decide", "output commands", "tool output controls", "tool result controls"]
SUMMARY_ONLY_DRIFT = ["conversation summary is memory", "memory is just a summary", "simple recap"]
LITERAL_SHAPE_DRIFT = ["circle", "square", "triangle", "geometric shape"]

def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text or ""))

def jaccard(a: str, b: str) -> float:
    wa = set(re.findall(r"\b[a-zA-Z]{3,}\b", normalize_text(a)))
    wb = set(re.findall(r"\b[a-zA-Z]{3,}\b", normalize_text(b)))
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

def audit_answer(answer: str, contract: RuntimeContract, origin: str = "model") -> Dict[str, Any]:
    a = normalize_text(answer)
    wc = word_count(answer)

    required_hits = count_terms(a, contract.required_terms)
    required_ratio = required_hits / max(1, len(contract.required_terms))
    operation_hits = count_terms(a, contract.required_operations)
    operation_ratio = operation_hits / max(1, len(contract.required_operations))

    forbidden_hits = [term for term in contract.forbidden_terms if term.lower() in a]

    legal_drift = contains_any(a, LEGAL_DRIFT_TERMS)
    admin_policy_drift = contract.profile == "evidence_control" and contains_any(a, ADMIN_POLICY_DRIFT)
    tool_command_drift = contract.profile == "evidence_control" and contains_any(a, TOOL_COMMAND_DRIFT)
    summary_only_drift = contract.profile == "memory_trace" and contains_any(a, SUMMARY_ONLY_DRIFT)
    keyword_only_drift = contract.profile == "inverse_retrieval" and ("keyword" in a and "operation" not in a and "function" not in a)
    literal_shape_drift = contract.profile == "inverse_retrieval" and contains_any(a, LITERAL_SHAPE_DRIFT) and not contains_any(a, ["operation", "function", "need", "slot", "candidate"])

    if contract.profile == "runtime_contract":
        checks = {
            "runtime_not_legal": not legal_drift and contains_any(a, ["runtime", "execution", "tool", "function", "api", "call"]),
            "preconditions": contains_any(a, ["precondition", "before", "prerequisite"]),
            "postconditions": contains_any(a, ["postcondition", "after", "verify", "expected state"]),
            "side_effects": contains_any(a, ["side effect", "bounded", "scope", "change"]),
            "rollback": contains_any(a, ["rollback", "restore", "safe failure", "recover"]),
        }
    elif contract.profile == "tool_safety":
        checks = {
            "permission": contains_any(a, ["permission", "authorized", "authorization", "allowed"]),
            "precondition": contains_any(a, ["precondition", "prerequisite", "before"]),
            "risk": contains_any(a, ["risk", "unsafe", "danger", "impact"]),
            "side_effect": contains_any(a, ["side effect", "bounded", "scope"]),
            "reject_or_safe_fail": contains_any(a, ["reject", "safe failure", "rollback", "abort"]),
        }
    elif contract.profile == "evidence_control":
        checks = {
            "evidence_not_command": contains_any(a, ["evidence", "observation", "inform"]) and not tool_command_drift,
            "controller": contains_any(a, ["controller", "agent", "runtime"]) and not admin_policy_drift,
            "policy_runtime": contains_any(a, ["policy", "decision rule", "gate", "contract"]) and not admin_policy_drift,
            "verify": contains_any(a, ["verify", "validate", "check", "corroborate"]),
            "next_action_owned": contains_any(a, ["next action", "decide", "interpret", "selection"]),
        }
    elif contract.profile == "state_recovery":
        checks = {
            "rollback_restore": contains_any(a, ["rollback", "restore", "revert", "prior valid state"]),
            "trace_preserved": contains_any(a, ["trace", "causal", "log", "history", "evidence"]),
            "failure_handled": contains_any(a, ["failed", "failure", "error", "corrupted", "bad side effect"]),
            "no_erasure": not contains_any(a, ["erase evidence", "delete trace", "forget"]),
            "cascade_prevented": contains_any(a, ["cascade", "prevent", "contain", "isolate", "safe"]),
        }
    elif contract.profile == "memory_trace":
        checks = {
            "not_summary": contains_any(a, ["not a summary", "more than a summary", "summary loses", "not just"]),
            "causal_trace": contains_any(a, ["trace", "causal", "which-path", "state transition"]),
            "observations_decisions": contains_any(a, ["observation", "decision", "action", "result"]),
            "update_continuity": contains_any(a, ["update", "continuity", "across turns", "prior state"]),
            "state_preserved": contains_any(a, ["state", "history", "event"]),
        }
    elif contract.profile == "inverse_retrieval":
        checks = {
            "operation": contains_any(a, ["operation", "function", "action", "transformation", "affordance"]),
            "missing_slot": contains_any(a, ["missing", "slot", "need", "inverse", "desired effect"]),
            "candidate_generation": contains_any(a, ["candidate", "generate", "retrieve", "search", "rank"]),
            "reject_noun_only": contains_any(a, ["reject", "not keyword", "not noun", "surface", "label"]),
            "verify_function": contains_any(a, ["verify", "preserve", "fit", "closes", "works"]),
        }
    elif contract.profile == "core_kernel":
        checks = {
            "runtime": contains_any(a, ["runtime", "inference", "control field"]),
            "not_weights": contains_any(a, ["not changing weights", "without changing", "inference-time", "runtime behavior"]),
            "contract": contains_any(a, ["contract", "profile", "need-slot"]),
            "branch_audit": contains_any(a, ["branch", "audit", "verify", "collapse"]),
            "trace": contains_any(a, ["trace", "continuity", "state"]),
        }
    else:
        checks = {"direct": wc >= 35, "specific": required_ratio > 0 or operation_ratio > 0, "no_bad_drift": not legal_drift and not tool_command_drift and not keyword_only_drift}

    profile_quality = sum(1 for v in checks.values() if v) / max(1, len(checks))

    prompt_requests_schema = contains_any(contract.prompt, ["build", "design", "contract", "checklist", "verifier", "schema"])
    schema_like = contains_any(a, ["precondition", "postcondition", "success criteria", "failure criteria"]) and wc < 90
    schema_echo_penalty = 0.0 if prompt_requests_schema else (0.08 if schema_like else 0.0)

    drift_flags = {
        "legal_drift": legal_drift,
        "admin_policy_drift": admin_policy_drift,
        "tool_command_drift": tool_command_drift,
        "summary_only_drift": summary_only_drift,
        "keyword_only_drift": keyword_only_drift,
        "literal_shape_drift": literal_shape_drift,
    }
    drift_penalties = 0.16 * sum(1 for v in drift_flags.values() if v) + 0.06 * len(forbidden_hits) + schema_echo_penalty

    length_score = min(1.0, max(0.0, wc / 80))
    if wc > 260:
        length_score -= min(0.25, (wc - 260) / 600)

    origin_score = 1.0 if origin == "model" else 0.0
    score = 0.38 * profile_quality + 0.20 * required_ratio + 0.12 * operation_ratio + 0.12 * length_score + 0.10 * (1.0 if not any(drift_flags.values()) else 0.0) + 0.08 * origin_score - drift_penalties
    score = float(max(0.0, min(1.0, score)))

    rejection_reasons = []
    if origin != "model":
        rejection_reasons.append("origin_not_model")
    if wc < 25:
        rejection_reasons.append("too_short")
    for k, v in drift_flags.items():
        if v:
            rejection_reasons.append(k)
    if forbidden_hits:
        rejection_reasons.append("forbidden_hits:" + ",".join(forbidden_hits))
    if profile_quality < 0.45:
        rejection_reasons.append("profile_quality_low")
    if score < PROFILE_THRESHOLDS.get(contract.profile, 0.72):
        rejection_reasons.append("below_profile_threshold")

    return {
        "score": score,
        "word_count": wc,
        "required_hits": required_hits,
        "required_ratio": required_ratio,
        "operation_hits": operation_hits,
        "operation_ratio": operation_ratio,
        "profile_quality": profile_quality,
        "profile_checks": checks,
        "forbidden_hits": forbidden_hits,
        "drift_flags": drift_flags,
        "schema_like": schema_like,
        "schema_echo_penalty": schema_echo_penalty,
        "prompt_requests_schema": prompt_requests_schema,
        "rejection_reasons": rejection_reasons,
    }

def should_reject(audit: Dict[str, Any], contract: RuntimeContract) -> bool:
    hard = {"legal_drift", "admin_policy_drift", "tool_command_drift", "summary_only_drift", "keyword_only_drift", "literal_shape_drift"}
    if any(r in hard for r in audit["rejection_reasons"]):
        return True
    return audit["score"] < PROFILE_THRESHOLDS.get(contract.profile, 0.72)

def consensus_signal(branches: List[BranchResult], contract: RuntimeContract) -> Dict[str, Any]:
    valid = [b for b in branches if not b.rejected and b.origin == "model"]
    if len(valid) < 2:
        return {"ok": False, "reason": "fewer_than_two_valid", "agreement": 0.0, "valid_count": len(valid)}

    top = sorted(valid, key=lambda b: b.score, reverse=True)[:3]
    check_keys = set()
    for b in top:
        check_keys |= set(b.audit.get("profile_checks", {}).keys())
    if not check_keys:
        op_agreement = 0.0
    else:
        agreed = 0
        for k in check_keys:
            vals = [bool(b.audit.get("profile_checks", {}).get(k, False)) for b in top]
            if sum(vals) >= 2:
                agreed += 1
        op_agreement = agreed / len(check_keys)

    text_agreement = np.mean([jaccard(top[i].answer, top[j].answer) for i in range(len(top)) for j in range(i+1, len(top))]) if len(top) > 1 else 0.0
    mean_score = float(np.mean([b.score for b in top]))

    ok = mean_score >= PROFILE_THRESHOLDS.get(contract.profile, 0.72) - 0.03 and op_agreement >= 0.60
    return {"ok": bool(ok), "reason": "consensus_operational_agreement" if ok else "consensus_insufficient", "valid_count": len(valid), "mean_score": mean_score, "op_agreement": float(op_agreement), "text_agreement": float(text_agreement)}

def collapse_gate(branches: List[BranchResult], contract: RuntimeContract) -> Tuple[str, str, Optional[BranchResult], Dict[str, Any]]:
    if not branches:
        return "Ω", "no_branches", None, {}
    ordered = sorted(branches, key=lambda b: b.score, reverse=True)
    best = ordered[0]
    second = ordered[1] if len(ordered) > 1 else None
    threshold = PROFILE_THRESHOLDS.get(contract.profile, 0.72)
    margin = best.score - (second.score if second else 0.0)
    consensus = consensus_signal(branches, contract)

    if best.origin == "model" and not best.rejected and best.score >= threshold and margin >= 0.055:
        return "Ψ", "direct_margin_collapse", best, {"margin": margin, "consensus": consensus}
    if consensus["ok"]:
        valid = [b for b in ordered if not b.rejected and b.origin == "model"]
        return "Ψ", "consensus_collapse", valid[0], {"margin": margin, "consensus": consensus}
    return "Ω", "max_depth_residue", best, {"margin": margin, "consensus": consensus}


## 7. Payload Shaper

In [ ]:

def shape_payload(prompt: str, contract: RuntimeContract, raw_answer: str, raw_score: float) -> Tuple[str, bool, Dict[str, Any]]:
    if not SHAPER_ENABLED:
        return raw_answer, False, {"reason": "shaper_disabled"}

    shape_prompt = f"""
Compress the accepted answer without changing its operational meaning.

USER PROMPT:
{prompt}

PRESERVED FUNCTION:
{contract.preserved_function}

SEMANTIC LOCKS:
{json.dumps(contract.semantic_locks, indent=2)}

RAW ACCEPTED ANSWER:
{raw_answer}

SHAPING RULES:
- Keep the answer direct and useful.
- Preserve required operations: {", ".join(contract.required_operations)}
- Do not introduce forbidden drift: {", ".join(contract.forbidden_terms)}
- Do not turn runtime terms into legal/organizational terms.
- Aim for 40-120 words unless a schema/checklist is needed.
""".strip()

    try:
        shaped = model_generate(shape_prompt, MAX_NEW_TOKENS_SHAPER, temperature=0.35)
        shaped_audit = audit_answer(shaped, contract, origin="model")
        raw_wc = word_count(raw_answer)
        shaped_wc = word_count(shaped)
        compression = 1 - (shaped_wc / max(1, raw_wc))
        accepted = (
            shaped_audit["score"] >= max(PROFILE_THRESHOLDS.get(contract.profile, 0.72) - 0.04, raw_score - 0.12)
            and shaped_wc <= max(raw_wc + 15, 140)
            and not any(shaped_audit["drift_flags"].values())
        )
        return (shaped if accepted else raw_answer), bool(accepted), {"reason": "accepted" if accepted else "rejected", "raw_words": raw_wc, "shaped_words": shaped_wc, "compression_ratio": compression, "shaped_audit": shaped_audit}
    except Exception:
        return raw_answer, False, {"reason": "shaper_error", "error": traceback.format_exc()}


## 8. Prompt Battery

In [ ]:

PROMPT_BATTERY = [
    # Runtime contract
    "explain why current AI agents fail when they use tools before forming a contract",
    "design a runtime contract for a file-writing tool",
    "explain success and failure criteria for an API call",
    "why should postconditions be checked after a tool runs",
    "build a tool-use contract for deleting a file",
    "show how preconditions and postconditions bound a function call",

    # Tool safety
    "how should an agent decide whether a tool call is safe",
    "define allowed side effects for a tool before calling it",
    "design a safety gate for an external API call",
    "when should an agent reject a tool call",
    "explain permission checks before tool execution",
    "describe safe failure for a dangerous tool action",

    # Evidence control
    "why is tool output evidence rather than the driver of the agent",
    "explain why a tool result should not control the next action by itself",
    "describe tool output as observation not command",
    "how should an agent treat conflicting tool outputs",
    "why should the controller own policy after a tool returns",
    "explain why evidence must pass through the contract gate",

    # State recovery
    "describe how rollback protects an agent after a failed API call",
    "explain how trace continuity helps an agent recover after a mistake",
    "how does rollback relate to memory in an agent runtime",
    "design a recovery path after a failed tool call",
    "explain restore previous state without erasing evidence",
    "why should rollback preserve the causal trace",

    # Memory trace
    "explain memory in an agent as trace continuity rather than a text summary",
    "why is a conversation summary not the same as agent memory",
    "describe agent memory as state transitions observations decisions and updates",
    "describe memory as causal event history across turns",
    "why does context amnesia break recursive agents",
    "explain why summaries lose which-path information",

    # Inverse retrieval
    "design a shape-first retrieval step where no noun match exists but the inverse need is clear",
    "how should retrieval work when keywords fail but the operation is obvious",
    "explain inverse operational fit for search without noun matching",
    "design a verifier for retrieval candidates selected by need rather than label",
    "how can an agent rank candidates by function instead of name",
    "build a retrieval step that rejects keyword-only matches",

    # Kernel meta
    "explain how a contract-aware runtime interface changes model behavior without changing model weights",
    "describe the difference between raw model inference and recursive collapse control",
    "explain why false Ψ is worse than honest Ω in an agent runtime",
    "design a trace-governed collapse gate for an AI agent",
    "explain why payload shaping must not damage the accepted collapse",
    "describe the RHI kernel as a model inside a recursive control field",
]

PROMPTS = PROMPT_BATTERY[:RUN_PROMPT_LIMIT]
print("Prompt count:", len(PROMPTS))


## 9. Core Kernel Execution

In [ ]:

def run_prompt(prompt: str) -> PromptResult:
    trace: List[TraceEvent] = []
    tick = 0

    def log(event_type: str, payload: Dict[str, Any]):
        nonlocal tick
        tick += 1
        trace.append(TraceEvent(t=tick, event_type=event_type, payload=payload))

    profile, profile_info = classify_task_profile(prompt)
    contract = compile_contract(prompt, profile)

    log("profile_selected", {"profile": profile, **profile_info})
    log("contract_compiled", asdict(contract))

    all_branches: List[BranchResult] = []
    state = "Ω"
    reason = "not_started"
    winner = None
    collapse_meta = {}
    prior_failure = None

    for depth in range(MAX_RECURSION_DEPTH + 1):
        log("depth_start", {"depth": depth})
        roles_this_depth = BRANCH_ROLES if depth == 0 else ["repair", "verify", "counter", "construct"]

        for role in roles_this_depth:
            try:
                bp = make_branch_prompt(prompt, contract, role, prior_failure=prior_failure)
                answer = model_generate(bp, MAX_NEW_TOKENS_BRANCH, temperature=TEMPERATURE)
                audit = audit_answer(answer, contract, origin="model")
                rejected = should_reject(audit, contract)
                br = BranchResult(role=role, origin="model", depth=depth, prompt_used=bp, answer=answer, score=audit["score"], audit=audit, rejected=rejected, rejection_reasons=audit["rejection_reasons"])
                all_branches.append(br)
                log("branch_generated", {"depth": depth, "role": role, "score": br.score, "rejected": br.rejected, "reasons": br.rejection_reasons, "answer_preview": answer[:180]})
            except Exception:
                err = traceback.format_exc()
                br = BranchResult(role=role, origin="error", depth=depth, prompt_used="", answer="", score=0.0, audit={"error": err}, rejected=True, rejection_reasons=["generation_error"])
                all_branches.append(br)
                log("branch_error", {"depth": depth, "role": role, "error": err})

        state, reason, winner, collapse_meta = collapse_gate(all_branches, contract)
        log("collapse_checked", {"depth": depth, "state": state, "reason": reason, "winner_role": winner.role if winner else None, "winner_score": winner.score if winner else None, "collapse_meta": collapse_meta})

        if state == "Ψ":
            break

        if winner:
            prior_failure = {"reason": reason, "missing": [k for k, v in winner.audit.get("profile_checks", {}).items() if not v], "forbidden_hits": winner.audit.get("forbidden_hits", []), "answer_preview": winner.answer[:180]}
        else:
            prior_failure = {"reason": reason, "missing": ["no winner"], "forbidden_hits": [], "answer_preview": ""}

    raw_answer = winner.answer if winner else ""
    final_answer = raw_answer
    shaped_answer = None
    shaping_attempted = False
    shaping_accepted = False
    shaper_meta = {}

    if state == "Ψ" and winner:
        shaping_attempted = True
        final_answer, shaping_accepted, shaper_meta = shape_payload(prompt, contract, raw_answer, winner.score)
        shaped_answer = final_answer if shaping_accepted else None
        log("payload_shaped", {"attempted": True, "accepted": shaping_accepted, "meta": shaper_meta})

    if REQUIRE_MODEL_FOR_PSI and not any(b.origin == "model" and b.answer for b in all_branches):
        state = "Ω"
        reason = "model_origin_required_no_model_output"
        final_answer = ""

    metrics = {
        "branch_count": len(all_branches),
        "rejected_branch_count": sum(1 for b in all_branches if b.rejected),
        "exhaust_ratio": sum(1 for b in all_branches if b.rejected) / max(1, len(all_branches)),
        "best_score": max([b.score for b in all_branches], default=0.0),
        "mean_score": float(np.mean([b.score for b in all_branches])) if all_branches else 0.0,
        "profile_threshold": PROFILE_THRESHOLDS.get(profile, 0.72),
        "collapse_meta": collapse_meta,
        "shaper_meta": shaper_meta,
    }

    return PromptResult(
        prompt=prompt,
        profile=profile,
        contract=asdict(contract),
        state=state,
        reason=reason,
        depth=max([b.depth for b in all_branches], default=0),
        winner_branch=winner.role if winner else None,
        winner_origin=winner.origin if winner else None,
        winner_score=winner.score if winner else 0.0,
        answer=final_answer,
        raw_winner_answer=raw_answer,
        shaped_answer=shaped_answer,
        shaping_attempted=shaping_attempted,
        shaping_accepted=shaping_accepted,
        branches=[asdict(b) for b in all_branches],
        trace=[asdict(t) for t in trace],
        metrics=metrics,
    )

if REQUIRE_MODEL_FOR_PSI and not MODEL_GENERATION_READY:
    raise RuntimeError("Model generation is not ready. v25 refuses fallback Ψ.")

t0 = time.time()
RESULTS: List[PromptResult] = []

for i, prompt in enumerate(PROMPTS, start=1):
    print(f"\n{'='*90}")
    print(f"[{i}/{len(PROMPTS)}] {prompt}")
    print("="*90)
    try:
        result = run_prompt(prompt)
    except Exception:
        err = traceback.format_exc()
        fallback_contract = compile_contract(prompt, "general")
        result = PromptResult(
            prompt=prompt, profile="general", contract=asdict(fallback_contract), state="Ω", reason="kernel_exception", depth=0,
            winner_branch=None, winner_origin=None, winner_score=0.0, answer="", raw_winner_answer="", shaped_answer=None,
            shaping_attempted=False, shaping_accepted=False, branches=[],
            trace=[asdict(TraceEvent(t=1, event_type="kernel_exception", payload={"error": err}))],
            metrics={"error": err},
        )
    RESULTS.append(result)
    print("STATE:", result.state, "REASON:", result.reason, "PROFILE:", result.profile, "WINNER:", result.winner_branch, result.winner_score)
    print("ANSWER PREVIEW:", result.answer[:240].replace("\n", " "))

elapsed_seconds = time.time() - t0
print("\nCompleted:", len(RESULTS), "Elapsed seconds:", elapsed_seconds)


## 10. Aggregate Metrics and Two-File Output

In [ ]:

def summarize_results(results: List[PromptResult]) -> Tuple[Dict[str, Any], pd.DataFrame]:
    rows = []
    for r in results:
        raw_words = word_count(r.raw_winner_answer)
        final_words = word_count(r.answer)
        rows.append({
            "run_id": RUN_ID,
            "prompt": r.prompt,
            "profile": r.profile,
            "state": r.state,
            "reason": r.reason,
            "depth": r.depth,
            "winner_branch": r.winner_branch,
            "winner_origin": r.winner_origin,
            "winner_score": r.winner_score,
            "branch_count": r.metrics.get("branch_count", 0),
            "rejected_branch_count": r.metrics.get("rejected_branch_count", 0),
            "exhaust_ratio": r.metrics.get("exhaust_ratio", 0.0),
            "mean_score": r.metrics.get("mean_score", 0.0),
            "profile_threshold": r.metrics.get("profile_threshold", None),
            "shaping_attempted": r.shaping_attempted,
            "shaping_accepted": r.shaping_accepted,
            "raw_words": raw_words,
            "final_words": final_words,
            "compression_ratio": 1 - final_words / max(1, raw_words),
            "answer_preview": (r.answer or "")[:240].replace("\n", " "),
        })

    df = pd.DataFrame(rows)

    aggregate = {
        "run_id": RUN_ID,
        "version": "v25",
        "purpose": "core_agent_kernel",
        "model_id_or_path": MODEL_ID_OR_PATH,
        "model_ready": MODEL_READY,
        "model_generation_ready": MODEL_GENERATION_READY,
        "model_error": MODEL_ERROR,
        "smoke_text": SMOKE_TEXT,
        "dependency_status": DEPENDENCY_STATUS,
        "device_info": DEVICE_INFO,
        "config": {
            "run_prompt_limit": RUN_PROMPT_LIMIT,
            "max_recursion_depth": MAX_RECURSION_DEPTH,
            "branch_roles": BRANCH_ROLES,
            "max_new_tokens_branch": MAX_NEW_TOKENS_BRANCH,
            "max_new_tokens_shaper": MAX_NEW_TOKENS_SHAPER,
            "temperature": TEMPERATURE,
            "shaper_enabled": SHAPER_ENABLED,
            "require_model_for_psi": REQUIRE_MODEL_FOR_PSI,
        },
        "total_prompts": len(results),
        "psi_count": int((df["state"] == "Ψ").sum()) if len(df) else 0,
        "omega_count": int((df["state"] == "Ω").sum()) if len(df) else 0,
        "psi_ratio": float((df["state"] == "Ψ").mean()) if len(df) else 0.0,
        "omega_ratio": float((df["state"] == "Ω").mean()) if len(df) else 0.0,
        "mean_winner_score": float(df["winner_score"].mean()) if len(df) else 0.0,
        "mean_exhaust_ratio": float(df["exhaust_ratio"].mean()) if len(df) else 0.0,
        "shaping_accepted_ratio": float(df["shaping_accepted"].mean()) if len(df) else 0.0,
        "mean_compression_ratio": float(df["compression_ratio"].mean()) if len(df) else 0.0,
        "elapsed_seconds": elapsed_seconds,
        "profile_metrics": {},
        "reason_counts": dict(Counter(df["reason"])) if len(df) else {},
    }

    if len(df):
        for profile, g in df.groupby("profile"):
            aggregate["profile_metrics"][profile] = {
                "count": int(len(g)),
                "psi_count": int((g["state"] == "Ψ").sum()),
                "omega_count": int((g["state"] == "Ω").sum()),
                "psi_ratio": float((g["state"] == "Ψ").mean()),
                "mean_winner_score": float(g["winner_score"].mean()),
                "mean_exhaust_ratio": float(g["exhaust_ratio"].mean()),
                "shaping_accepted_ratio": float(g["shaping_accepted"].mean()),
                "mean_compression_ratio": float(g["compression_ratio"].mean()),
                "reason_counts": dict(Counter(g["reason"])),
            }

    return aggregate, df

aggregate, summary_df = summarize_results(RESULTS)

bundle = {
    "run_id": RUN_ID,
    "version": "v25",
    "purpose": "core_agent_kernel",
    "created_root": str(ROOT),
    "out_dir": str(OUT_DIR),
    "aggregate": aggregate,
    "summary": summary_df.to_dict(orient="records"),
    "results": [asdict(r) for r in RESULTS],
    "profile_thresholds": PROFILE_THRESHOLDS,
    "profile_lexicons": PROFILE_LEXICONS,
    "semantic_locks": SEMANTIC_LOCKS,
    "interpretation_lock": {
        "main_branch": "new AI runtime / RHI core kernel",
        "not_this": ["H-probing", "raw benchmark detour", "prompt-only wrapper"],
        "core_claim": "model behavior is changed at inference time by recursive contract field, branch audit, trace continuity, and Ψ/Ω collapse control",
        "psi_rule": "Ψ requires model-origin answer preserving task-local operation",
        "omega_rule": "Ω is preferred over false operational collapse",
    },
}

bundle_out = OUT_DIR / f"{RUN_ID}_bundle.json"
summary_out = OUT_DIR / f"{RUN_ID}_summary.csv"

with open(bundle_out, "w", encoding="utf-8") as f:
    json.dump(bundle, f, indent=2, ensure_ascii=False)

summary_df.to_csv(summary_out, index=False)

print("Saved exactly two output files:")
print(bundle_out)
print(summary_out)
print("\nAggregate:")
print(json.dumps(aggregate, indent=2, ensure_ascii=False)[:4000])
display(summary_df)


## 11. Reading v25

The important output is not one score. It is whether the kernel behaves like a runtime:

$$
\boxed{
\text{profile} \rightarrow \text{contract} \rightarrow \text{branch} \rightarrow \text{audit} \rightarrow \Psi/\Omega \rightarrow \text{trace}
}
$$

A good v25 run should show:

- no model fallback,
- no dependency tear,
- low junk `general` routing,
- $\Omega$ where semantic carrier drift remains,
- $\Psi$ where task-local operation is preserved,
- shaper accepted only when meaning survives compression.

This is the new AI branch.
